In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors
import rasterio
from rasterio.warp import reproject, Resampling
from matplotlib.patches import Patch

plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'Liberation Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Arial', 'DejaVu Serif', 'Liberation Serif']
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans', 'Liberation Sans']

plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 10
plt.rcParams['figure.titlesize'] = 16

plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 600

QUADRANTS = [
    (r"Ts+,$R_{max}$+", lambda X, Y: (X >= 0) & (Y >= 0), "pos_pos"),
    (r"Ts-,$R_{max}$+", lambda X, Y: (X < 0) & (Y >= 0), "neg_pos"),
    (r"Ts-,$R_{max}$-", lambda X, Y: (X < 0) & (Y < 0), "neg_neg"),
    (r"Ts+,$R_{max}$-", lambda X, Y: (X >= 0) & (Y < 0), "pos_neg"),
]

def calculate_latitude_weights(lat):
    lat_rad = np.deg2rad(lat)
    return np.abs(np.cos(lat_rad))

def create_weight_matrix(lon, lat):
    lat_weights = calculate_latitude_weights(lat)
    return np.broadcast_to(lat_weights[:, np.newaxis], (len(lat), len(lon)))

def load_and_resample_mask(mask_path, target_lon, target_lat):
    with rasterio.open(mask_path) as src:
        mask_data = src.read(1)
        mask_transform = src.transform
        mask_crs = src.crs

    from rasterio.transform import from_bounds
    target_transform = from_bounds(
        target_lon.min(), target_lat.min(),
        target_lon.max(), target_lat.max(),
        len(target_lon), len(target_lat)
    )

    target_mask = np.zeros((len(target_lat), len(target_lon)), dtype=np.float32)

    reproject(
        source=mask_data,
        destination=target_mask,
        src_transform=mask_transform,
        src_crs=mask_crs,
        dst_transform=target_transform,
        dst_crs='EPSG:4326',
        resampling=Resampling.nearest
    )

    mask_bool = (target_mask > 0) & (~np.isnan(target_mask))
    mask_bool = np.flipud(mask_bool)
    return mask_bool

def to_rgb(c):
    if isinstance(c, str):
        return np.array(mcolors.to_rgb(c), dtype=float)
    return np.array(c, dtype=float)

def bivariate_cmap_endpoints(X, Y, endpoints, gamma=1.0, robust=95, nodata_color=(1, 1, 1)):
    valid = ~(np.isnan(X) | np.isnan(Y))

    X0 = np.where(valid, X, 0.0)
    Y0 = np.where(valid, Y, 0.0)

    Xv = X0[valid]
    Yv = Y0[valid]
    if Xv.size > 0:
        X_max = np.percentile(np.abs(Xv), robust)
        Y_max = np.percentile(np.abs(Yv), robust)
        X_max = max(X_max, 1e-12)
        Y_max = max(Y_max, 1e-12)
    else:
        X_max = Y_max = 1.0

    Xn = np.clip(X0 / X_max, -1, 1)
    Yn = np.clip(Y0 / Y_max, -1, 1)

    img = np.ones(X.shape + (3,), dtype=float)

    def paint(mask2, key):
        if not np.any(mask2):
            return
        strength = np.sqrt(Xn[mask2]**2 + Yn[mask2]**2) / np.sqrt(2)
        strength = np.clip(strength, 0, 1)
        if gamma != 1.0:
            strength = strength**gamma

        light = to_rgb(endpoints[key]['light'])
        dark = to_rgb(endpoints[key]['dark'])
        img[mask2] = (1 - strength)[:, None] * light + strength[:, None] * dark

    paint((Xn >= 0) & (Yn >= 0) & valid, 'pos_pos')
    paint((Xn >= 0) & (Yn < 0) & valid, 'pos_neg')
    paint((Xn < 0) & (Yn >= 0) & valid, 'neg_pos')
    paint((Xn < 0) & (Yn < 0) & valid, 'neg_neg')

    img[~valid] = np.array(nodata_color, dtype=float)
    return img

def add_significance_stipple_points(
    ax, lon, lat, X_sig, Y_sig,
    step=6, s=10, s_both=16,
    alpha=0.70,
    ts_color='0.55',
    mc_color='#7B2CBF',
    both_color='k',
    zorder=6
):
    Lon, Lat = np.meshgrid(lon, lat)

    step = int(step)
    sl = (slice(None, None, step), slice(None, None, step))
    Lon_s = Lon[sl]
    Lat_s = Lat[sl]
    Xs = X_sig[sl]
    Ys = Y_sig[sl]

    both = Xs & Ys
    only_x = Xs & (~Ys)
    only_y = Ys & (~Xs)

    if np.any(only_x):
        ax.scatter(
            Lon_s[only_x], Lat_s[only_x],
            s=s, c=ts_color, marker='.',
            alpha=alpha, linewidths=0,
            transform=ccrs.PlateCarree(), zorder=zorder
        )

    if np.any(only_y):
        ax.scatter(
            Lon_s[only_y], Lat_s[only_y],
            s=s, c=mc_color, marker='.',
            alpha=alpha, linewidths=0,
            transform=ccrs.PlateCarree(), zorder=zorder
        )

    if np.any(both):
        ax.scatter(
            Lon_s[both], Lat_s[both],
            s=s_both, c=both_color, marker='.',
            alpha=min(0.90, alpha + 0.10), linewidths=0,
            transform=ccrs.PlateCarree(), zorder=zorder + 1
        )

def weighted_median(values, weights):
    v = np.asarray(values, dtype=float).ravel()
    w = np.asarray(weights, dtype=float).ravel()

    good = (~np.isnan(v)) & (~np.isnan(w)) & (w > 0)
    if not np.any(good):
        return np.nan

    v = v[good]
    w = w[good]

    idx = np.argsort(v)
    v_sorted = v[idx]
    w_sorted = w[idx]

    cw = np.cumsum(w_sorted)
    cutoff = 0.5 * np.sum(w_sorted)
    j = np.searchsorted(cw, cutoff, side='left')
    j = np.clip(j, 0, len(v_sorted) - 1)
    return v_sorted[j]

def weighted_mean(values, weights):
    v = np.asarray(values, dtype=float).ravel()
    w = np.asarray(weights, dtype=float).ravel()

    good = (~np.isnan(v)) & (~np.isnan(w)) & (w > 0)
    if not np.any(good):
        return np.nan

    v = v[good]
    w = w[good]

    return np.sum(v * w) / np.sum(w)

def add_weighted_significance_bar_chart(
    fig,
    X_slope_masked, Y_slope_masked,
    X_pvalue_masked, Y_pvalue_masked,
    weight_matrix, mask, endpoints,
    position=(0.05, 0.05), width=0.18, height=0.4,
    show_legend=True,
    annotate_total=True,
    annotate_sig=False
):
    valid_mask = mask & ~np.isnan(X_slope_masked) & ~np.isnan(Y_slope_masked)
    if not np.any(valid_mask):
        return None, None

    total_weight = np.sum(weight_matrix[valid_mask])
    if total_weight <= 0:
        return None, None

    X_sig = (X_pvalue_masked < 0.05) & ~np.isnan(X_pvalue_masked)
    Y_sig = (Y_pvalue_masked < 0.05) & ~np.isnan(Y_pvalue_masked)
    both_sig = X_sig & Y_sig

    quadrant_data = []
    for name, cond, ckey in QUADRANTS:
        q_mask = cond(X_slope_masked, Y_slope_masked) & valid_mask

        total_q_weight = np.sum(weight_matrix[q_mask]) if np.any(q_mask) else 0.0
        sig_q_weight = np.sum(weight_matrix[q_mask & both_sig]) if np.any(q_mask & both_sig) else 0.0
        nonsig_q_weight = total_q_weight - sig_q_weight

        total_q_pct = total_q_weight / total_weight * 100
        sig_q_pct = sig_q_weight / total_weight * 100
        nonsig_q_pct = nonsig_q_weight / total_weight * 100

        quadrant_data.append({
            'name': name,
            'color_key': ckey,
            'total_pct': total_q_pct,
            'sig_pct': sig_q_pct,
            'nonsig_pct': nonsig_q_pct
        })

    ax_bar = fig.add_axes([position[0], position[1], width, height])
    y_pos = np.arange(len(quadrant_data))

    ax_bar.barh(
        y_pos,
        [q['total_pct'] for q in quadrant_data],
        color=[endpoints[q['color_key']]['dark'] for q in quadrant_data],
        alpha=0.95, edgecolor='white', linewidth=1
    )

    ax_bar.set_yticks(y_pos)
    ax_bar.set_yticklabels([q['name'] for q in quadrant_data], fontsize=9)
    ax_bar.set_xlabel('', fontsize=9)
    ax_bar.set_title('', fontsize=11, weight='bold', pad=6)

    if annotate_total:
        for i, q in enumerate(quadrant_data):
            if q['name'] == r"Ts-,$R_{max}$-":
                total_x_pos = q['total_pct'] + 2.0
            elif q['name'] == r"Ts-,$R_{max}$+":
                total_x_pos = q['total_pct'] + 1.0
            elif q['total_pct'] < 15:
                total_x_pos = q['total_pct'] + 0.8
            else:
                total_x_pos = q['total_pct'] + 0.5

            ax_bar.text(
                total_x_pos, i,
                f"{q['total_pct']:.1f}%",
                ha='left', va='center',
                fontsize=8.5, weight='bold', color='0.1'
            )

    max_pct = max(q['total_pct'] for q in quadrant_data) if quadrant_data else 1
    ax_bar.set_xlim(0, max_pct * 1.6)

    ax_bar.spines['top'].set_visible(False)
    ax_bar.spines['right'].set_visible(False)
    ax_bar.grid(axis='x', alpha=0.25, linestyle='--')
    ax_bar.tick_params(axis='x', labelsize=8)
    ax_bar.tick_params(axis='y', labelsize=9)
    ax_bar.invert_yaxis()

    return ax_bar, quadrant_data

def add_dual_axis_lollipop_inset(
    fig,
    X_slope_masked, Y_slope_masked,
    X_pvalue_masked, Y_pvalue_masked,
    weight_matrix, mask,
    endpoints,
    position=(0.76, 0.02), width=0.22, height=0.18,
    use_weighted_mean=True,
    ts_range=(-1.8, 1.8),
    ts_tick_value=1.0,
    mc_tick_value=0.05,
    show_both_sig=False
):
    ax = fig.add_axes([position[0], position[1], width, height])

    valid_mask = mask & ~np.isnan(X_slope_masked) & ~np.isnan(Y_slope_masked)
    if not np.any(valid_mask):
        ax.set_axis_off()
        return ax, None

    X_sig = (X_pvalue_masked < 0.05) & ~np.isnan(X_pvalue_masked)
    Y_sig = (Y_pvalue_masked < 0.05) & ~np.isnan(Y_pvalue_masked)
    both_sig = X_sig & Y_sig

    rows = []
    for name, cond, ckey in QUADRANTS:
        qmask = cond(X_slope_masked, Y_slope_masked) & valid_mask

        if not np.any(qmask):
            ts_stat = np.nan
            mc_stat = np.nan
            ts_stat_sig = np.nan
            mc_stat_sig = np.nan
        else:
            if use_weighted_mean:
                ts_stat = weighted_mean(X_slope_masked[qmask], weight_matrix[qmask])
                mc_stat = weighted_mean(Y_slope_masked[qmask], weight_matrix[qmask])
                if show_both_sig:
                    qs = qmask & both_sig
                    ts_stat_sig = weighted_mean(X_slope_masked[qs], weight_matrix[qs]) if np.any(qs) else np.nan
                    mc_stat_sig = weighted_mean(Y_slope_masked[qs], weight_matrix[qs]) if np.any(qs) else np.nan
                else:
                    ts_stat_sig = np.nan
                    mc_stat_sig = np.nan
            else:
                ts_stat = np.nanmean(X_slope_masked[qmask])
                mc_stat = np.nanmean(Y_slope_masked[qmask])
                if show_both_sig:
                    qs = qmask & both_sig
                    ts_stat_sig = np.nanmean(X_slope_masked[qs]) if np.any(qs) else np.nan
                    mc_stat_sig = np.nanmean(Y_slope_masked[qs]) if np.any(qs) else np.nan
                else:
                    ts_stat_sig = np.nan
                    mc_stat_sig = np.nan

        rows.append({
            'name': name,
            'color_key': ckey,
            'ts_mean': ts_stat,
            'mc_mean': mc_stat,
            'ts_mean_sig': ts_stat_sig,
            'mc_mean_sig': mc_stat_sig
        })

    ax.set_ylim(ts_range)
    ax.axhline(0, color='0.25', lw=1)
    ax.set_yticks([-ts_tick_value, ts_tick_value])
    ax.set_yticklabels([f'{-ts_tick_value:.0f}', f'{ts_tick_value:.0f}'])

    ax2 = ax.twinx()
    scale_factor = ts_range[1] / ts_tick_value
    mc_range_max = mc_tick_value * scale_factor
    mc_range = (-mc_range_max, mc_range_max)

    ax2.set_ylim(mc_range)
    ax2.axhline(0, color='0.25', lw=1)
    ax2.set_yticks([-mc_tick_value, mc_tick_value])
    ax2.set_yticklabels([f'{-mc_tick_value:.2f}', f'{mc_tick_value:.2f}'])

    x = np.arange(len(rows))
    ax.set_xticks([])
    ax.set_xticklabels([])

    dx = 0.14
    for i, r in enumerate(rows):
        xi = x[i]
        c_dark = endpoints[r['color_key']]['dark']
        c_light = endpoints[r['color_key']]['light']

        if np.isfinite(r['ts_mean']):
            ax.vlines(xi - dx, 0, r['ts_mean'], color=c_dark, lw=2.0, alpha=0.95)
            ax.scatter([xi - dx], [r['ts_mean']], s=34, color=c_dark,
                       edgecolor='white', linewidth=0.6, zorder=5)

        if np.isfinite(r['mc_mean']):
            ax2.vlines(xi + dx, 0, r['mc_mean'], color=c_light, lw=2.0, alpha=0.95)
            ax2.scatter([xi + dx], [r['mc_mean']], s=34, color=c_light,
                        edgecolor='white', linewidth=0.6, zorder=5)

        if show_both_sig and np.isfinite(r['ts_mean_sig']):
            ax.scatter([xi - dx], [r['ts_mean_sig']], s=48, facecolors='none',
                       edgecolors='k', linewidth=1.0, zorder=6)
        if show_both_sig and np.isfinite(r['mc_mean_sig']):
            ax2.scatter([xi + dx], [r['mc_mean_sig']], s=48, facecolors='none',
                        edgecolors='k', linewidth=1.0, zorder=6)

    ax.grid(axis='y', alpha=0.25, linestyle='--')

    ax.spines['top'].set_visible(True)
    ax.spines['bottom'].set_visible(True)
    ax.spines['left'].set_visible(True)
    ax.spines['right'].set_visible(True)

    for spine in ax.spines.values():
        spine.set_linewidth(0.8)
        spine.set_color('black')

    ax.tick_params(axis='y', labelsize=8, left=True, labelleft=True, right=False, labelright=False)
    ax2.tick_params(axis='y', labelsize=8, left=False, labelleft=False, right=True, labelright=True)

    ax.set_ylabel("Ts", fontsize=9, color='0.2', rotation=0, labelpad=15)
    ax.yaxis.set_label_coords(-0.2, 0.4)

    ax2.set_ylabel("$R_{max}$", fontsize=9, color='0.2', rotation=0, labelpad=15)
    ax2.yaxis.set_label_coords(1.3, 0.6)

    ax.set_title("Theil–Sen slope", weight='bold', fontsize=9, pad=6)
    ax.set_xlim(-0.5, len(rows) - 0.5)

    return ax, rows

def print_significance_weighted_statistics(X_slope_masked, Y_slope_masked, X_pvalue_masked, Y_pvalue_masked,
                                           weight_matrix, mask, quadrant_data):
    valid_mask = mask & ~np.isnan(X_slope_masked) & ~np.isnan(Y_slope_masked)
    total_weight = np.sum(weight_matrix[valid_mask])

    X_sig = (X_pvalue_masked < 0.05) & ~np.isnan(X_pvalue_masked)
    Y_sig = (Y_pvalue_masked < 0.05) & ~np.isnan(Y_pvalue_masked)
    both_sig = X_sig & Y_sig

    print(f"\nWeighted significance statistics:")
    print(f"Valid points inside mask: {np.sum(valid_mask)}")
    print(f"Total masked weight: {total_weight:.6f}")
    if total_weight > 0:
        print(f"Both-significant weight ratio: {np.sum(weight_matrix[both_sig & valid_mask]) / total_weight * 100:.1f}%")

    print(f"\nQuadrant weighted distribution:")
    print(f"{'Quadrant':<15} {'Total':<8} {'Sig':<8} {'Non-sig':<10}")
    print("-" * 55)

    for q in quadrant_data:
        print(f"{q['name']:<15} {q['total_pct']:>6.1f}% {q['sig_pct']:>8.1f}% {q['nonsig_pct']:>10.1f}%")

    total_all = sum(q['total_pct'] for q in quadrant_data)
    total_sig = sum(q['sig_pct'] for q in quadrant_data)
    total_nonsig = sum(q['nonsig_pct'] for q in quadrant_data)

    print("-" * 55)
    print(f"{'Total':<15} {total_all:>6.1f}% {total_sig:>8.1f}% {total_nonsig:>10.1f}%")

def print_dual_lollipop_values(lolli_rows):
    if lolli_rows is None:
        print("\nDual-axis lollipop data is empty")
        return

    print(f"\nDual-axis lollipop values (weighted mean):")
    print(f"{'Quadrant':<20} {'TS mean':<12} {'MC mean':<12}")
    print("-" * 50)

    for r in lolli_rows:
        ts_val = f"{r['ts_mean']:.6f}" if np.isfinite(r['ts_mean']) else "NaN"
        mc_val = f"{r['mc_mean']:.6f}" if np.isfinite(r['mc_mean']) else "NaN"
        print(f"{r['name']:<20} {ts_val:<12} {mc_val:<12}")

    print("-" * 50)

    ts_vals = [r['ts_mean'] for r in lolli_rows if np.isfinite(r['ts_mean'])]
    mc_vals = [r['mc_mean'] for r in lolli_rows if np.isfinite(r['mc_mean'])]

    if ts_vals:
        print(f"TS range: {min(ts_vals):.6f} to {max(ts_vals):.6f}")
    if mc_vals:
        print(f"MC range: {min(mc_vals):.6f} to {max(mc_vals):.6f}")
        print("TS ±1 corresponds to MC ±0.05")

output_dir = r"PATH_TO_OUTPUT_DIR"
mask_path = r"PATH_TO_MASK_FILE"

ds_time_scale = xr.open_dataset(f"{output_dir}\\time_scale_trend_analysis_Spring_full11.nc")
ds_max_corr = xr.open_dataset(f"{output_dir}\\max_correlation_trend_analysis_Spring_full11.nc")

X_slope = ds_time_scale['theil_slope'].values
Y_slope = ds_max_corr['theil_slope'].values
X_pvalue = ds_time_scale['mk_pvalue'].values
Y_pvalue = ds_max_corr['mk_pvalue'].values

lon = ds_time_scale.coords['lon'].values
lat = ds_time_scale.coords['lat'].values

weight_matrix = create_weight_matrix(lon, lat)

mask = load_and_resample_mask(mask_path, lon, lat)

X_slope_masked = X_slope.copy()
Y_slope_masked = Y_slope.copy()
X_pvalue_masked = X_pvalue.copy()
Y_pvalue_masked = Y_pvalue.copy()

X_slope_masked[~mask] = np.nan
Y_slope_masked[~mask] = np.nan
X_pvalue_masked[~mask] = np.nan
Y_pvalue_masked[~mask] = np.nan

X_significant = (X_pvalue_masked < 0.05) & ~np.isnan(X_pvalue_masked) & mask
Y_significant = (Y_pvalue_masked < 0.05) & ~np.isnan(Y_pvalue_masked) & mask

endpoints = {
    'pos_pos': {'light': "#D4D5F8", 'dark': "#626FB3"},
    'pos_neg': {'light': "#B7E6FA", 'dark': "#059ECC"},
    'neg_pos': {'light': "#FFB9B9FF", 'dark': "#FF0000FF"},
    'neg_neg': {'light': "#D6E6CB", 'dark': "#1DA51D"},
}

fig = plt.figure(figsize=(7, 4))
ax = plt.axes(projection=ccrs.PlateCarree())

img = bivariate_cmap_endpoints(
    X_slope_masked, Y_slope_masked,
    endpoints=endpoints,
    gamma=1.0,
    robust=95,
    nodata_color=(1, 1, 1)
)

dlon = abs(lon[1] - lon[0]) if len(lon) > 1 else 0.1
dlat = abs(lat[1] - lat[0]) if len(lat) > 1 else 0.1

ax.imshow(
    img, origin='lower',
    extent=[lon.min(), lon.max(), lat.min(), lat.max()],
    transform=ccrs.PlateCarree(),
    interpolation='nearest',
    zorder=1
)

add_significance_stipple_points(
    ax, lon, lat, X_significant, Y_significant,
    step=6, s=10, s_both=18,
    alpha=0.75,
    ts_color='0.55',
    mc_color='#7B2CBF',
    both_color='k',
    zorder=6
)

ax.coastlines('110m', linewidth=0.8, color='black', zorder=10)
ax.add_feature(cfeature.BORDERS, linewidth=0.5, color='gray', zorder=10)
ax.add_feature(cfeature.LAND, facecolor='lightgray', alpha=0.2, zorder=0)
ax.add_feature(cfeature.OCEAN, facecolor='lightblue', alpha=0.1, zorder=0)

ax.set_extent(
    [lon.min() - dlon * 0.5, lon.max() + dlon * 0.5,
     lat.min() - dlat * 0.5, lat.max() + dlat * 0.5],
    crs=ccrs.PlateCarree()
)

gl = ax.gridlines(
    crs=ccrs.PlateCarree(), draw_labels=True,
    linewidth=0.5, color='gray', alpha=0.5, linestyle='--'
)
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {'size': 13, 'family': 'serif'}
gl.ylabel_style = {'size': 13, 'family': 'serif'}

ax_bar, quadrant_data = add_weighted_significance_bar_chart(
    fig,
    X_slope_masked, Y_slope_masked,
    X_pvalue_masked, Y_pvalue_masked,
    weight_matrix, mask, endpoints,
    position=(0.05, 0.05),
    width=0.18,
    height=0.4,
    show_legend=True,
    annotate_total=True,
    annotate_sig=False
)

if ax_bar is not None:
    ax_bar.set_position([0.232, 0.28, 0.09, 0.25])

ax_lolli, lolli_rows = add_dual_axis_lollipop_inset(
    fig,
    X_slope_masked, Y_slope_masked,
    X_pvalue_masked, Y_pvalue_masked,
    weight_matrix, mask,
    endpoints,
    position=(0.62, 0.06),
    width=0.20,
    height=0.16,
    use_weighted_mean=True,
    ts_range=(-1.8, 1.8),
    ts_tick_value=1.0,
    mc_tick_value=0.05,
    show_both_sig=False
)

if ax_lolli is not None:
    ax_lolli.set_position([0.58, 0.18, 0.2, 0.15])

bivariate_map_path = f"{output_dir}\\bivariate_trend_map_weighted_mean_aligned_Spring.png"
plt.savefig(
    bivariate_map_path,
    dpi=600,
    bbox_inches='tight',
    pad_inches=0.2,
    facecolor='white',
    edgecolor='none',
    format='png',
    metadata={'Software': 'matplotlib'}
)
plt.show()

if quadrant_data is not None:
    print_significance_weighted_statistics(
        X_slope_masked, Y_slope_masked,
        X_pvalue_masked, Y_pvalue_masked,
        weight_matrix, mask,
        quadrant_data
    )

print_dual_lollipop_values(lolli_rows)

ds_time_scale.close()
ds_max_corr.close()
